# Week 45, Day 5: Tree problems

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w045d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def leaf_sums(node, so_far, out):
    if node is None:
        return out
    so_far += node.val
    if node.left is None and node.right is None:
        out.append(so_far)
    leaf_sums(node.left, so_far, out)
    leaf_sums(node.right, so_far, out)
    return out

T = TreeNode
t = T(5, T(4, T(11, T(7), T(2))), T(8, T(13), T(4, None, T(1))))
print(leaf_sums(t, 0, []))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def paths_with_sum(root, target):
    found, path = [], []
    def walk(node, remaining):
        if node is None:
            return
        path.append(node.val)
        remaining -= node.val
        if node.left is None and node.right is None and remaining == 0:
            found.append(list(path))          # a copy: path keeps changing
        walk(node.left, remaining)
        walk(node.right, remaining)
        path.pop()                            # backtrack: undo this node before returning
    walk(root, target)
    return found

def good_nodes(root):
    """Nodes whose value is at least every value above them."""
    def walk(node, highest):
        if node is None:
            return 0
        good = 1 if node.val >= highest else 0
        highest = max(highest, node.val)
        return good + walk(node.left, highest) + walk(node.right, highest)
    return walk(root, float("-inf"))

T = TreeNode
t = T(5, T(4, T(11, T(7), T(2))), T(8, T(13), T(4, T(5), T(1))))
print(paths_with_sum(t, 22), good_nodes(T(3, T(1, T(3)), T(4, T(1), T(5)))))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def most_frequent_subtree_sum(root):
    counts = {}
    def total(node):
        if node is None:
            return 0
        s = node.val + total(node.left) + total(node.right)
        counts[s] = counts.get(s, 0) + 1
        return s
    total(root)
    best = max(counts.values())
    return sorted(s for s, c in counts.items() if c == best)

def count_unival(root):
    """Subtrees in which every value is the same."""
    count = [0]
    def same(node):
        """Is the subtree under node all one value? (Counts as it goes.)"""
        if node is None:
            return True
        left, right = same(node.left), same(node.right)        # always visit both
        ok = left and right and all(c is None or c.val == node.val for c in (node.left, node.right))
        if ok:
            count[0] += 1
        return ok
    same(root)
    return count[0]

T = TreeNode
print(most_frequent_subtree_sum(T(5, T(2), T(-5))), count_unival(T(5, T(1, T(5), T(5)), T(5, None, T(5)))))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def sum_numbers(node, so_far=0):
    if node is None:
        return 0
    so_far = so_far * 10 + node.val
    if node.left is None and node.right is None:
        return so_far
    return sum_numbers(node.left, so_far) + sum_numbers(node.right, so_far)

T = TreeNode
print(sum_numbers(T(4, T(9, T(5), T(1)), T(0))))       # 495 + 491 + 40

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def save(node):
    if node is None:
        return ",#"
    return f",{node.val}{save(node.left)}{save(node.right)}"

def is_subtree(big, small):
    return save(small) in save(big)

T = TreeNode
big = T(3, T(4, T(1), T(2)), T(5))
print(is_subtree(big, T(4, T(1), T(2))), is_subtree(big, T(4, T(1))), save(T(4, T(1), T(2))))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def count_paths(root, k):
    seen = {0: 1}                         # the empty prefix, above the root
    def walk(node, current):
        if node is None:
            return 0
        current += node.val
        found = seen.get(current - k, 0)
        seen[current] = seen.get(current, 0) + 1
        found += walk(node.left, current) + walk(node.right, current)
        seen[current] -= 1                # backtrack: this prefix leaves the path
        return found
    return walk(root, 0)

T = TreeNode
t = T(10, T(5, T(3, T(3), T(-2)), T(2, None, T(1))), T(-3, None, T(11)))
print(count_paths(t, 8))

> **Key idea.** Tree problems split by which way information flows. Down: pass the path's state as parameters (sums, depth, the highest value above); with a mutable path, append before recursing and pop after (backtracking), and copy it when you save it. Up: return what the parent needs from each subtree (post-order), visiting both children before combining. Both: parameters down, return values up. Compare whole trees with a recursive `same`, or by serialising with markers and checking text containment. Count downward paths with sum k in O(n) using a dictionary of prefix sums on the current path, removing each prefix when its node is finished.

### Exercise: Path sums

Write `has_path_sum(root, target)`: is there a root-to-**leaf** path whose values add up to `target`? (An empty tree has none.) Then `path_sums(root)`: the sum of every root-to-leaf path, left to right.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def has_path_sum(root, target):
    ___

def path_sums(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_paths():\n    \"path sums\"\n    t = T(5, T(4, T(11, T(7), T(2))), T(8, T(13), T(4, None, T(1))))\n    assert has_path_sum(t, 22) and not has_path_sum(t, 9) and not has_path_sum(None, 0)\n    assert path_sums(t) == [27, 22, 26, 18] and path_sums(None) == []\n    assert not has_path_sum(T(1, T(2)), 1), \"the root alone isn't a leaf here\"")

### Exercise: Good nodes

A node is **good** if no value on the path from the root down to it is bigger than its own. Write `good_nodes(root)` (how many) and `good_values(root)` (their values, in pre-order).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def good_nodes(root):
    ___

def good_values(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_good():\n    \"good nodes\"\n    t = T(3, T(1, T(3)), T(4, T(1), T(5)))\n    assert good_nodes(t) == 4 and good_values(t) == [3, 3, 4, 5]\n    assert good_nodes(T(3, T(3, T(4), T(2)))) == 3 and good_nodes(None) == 0 and good_values(None) == []")

### Exercise: Numbers along the paths

Each node holds a digit from 0 to 9, and each root-to-leaf path spells a number (1 → 2 → 3 is 123). Write `path_numbers(root)` (the numbers, left to right) and `sum_numbers(root)` (their total).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def path_numbers(root):
    ___

def sum_numbers(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_numbers():\n    \"numbers\"\n    t = T(4, T(9, T(5), T(1)), T(0))\n    assert path_numbers(t) == [495, 491, 40] and sum_numbers(t) == 1026\n    assert path_numbers(T(1, T(2), T(3))) == [12, 13] and sum_numbers(None) == 0 and path_numbers(T(0)) == [0]")

### Exercise: Single-value subtrees

Write `count_unival(root)`: how many subtrees have every node holding the same value (every leaf counts as one). Visit both children before deciding, so nothing is skipped. Then `largest_unival(root)`: the size of the biggest such subtree (0 for an empty tree).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def count_unival(root):
    ___

def largest_unival(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_unival():\n    \"single-value subtrees\"\n    t = T(5, T(1, T(5), T(5)), T(5, None, T(5)))\n    assert count_unival(t) == 4 and largest_unival(t) == 2\n    u = T(1, T(1, T(1), T(1)), T(1, T(1)))\n    assert count_unival(u) == 6 and largest_unival(u) == 6 and count_unival(None) == 0 and largest_unival(None) == 0\n    v = T(2, T(3, T(3), T(3)), T(2, T(2), T(9)))\n    assert count_unival(v) == 5 and largest_unival(v) == 3")

### Exercise: Is it a subtree?

Write `is_subtree(big, small)`: does `big` contain a node whose whole subtree is identical (same shape and values) to `small`? An empty `small` is a subtree of anything. Do it in O(n + m) by serialising both trees in pre-order with markers for missing children and separators around every value, then checking containment. The checks use trees of 20,000 nodes.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def is_subtree(big, small):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef test_small():\n    \"subtrees\"\n    big = T(3, T(4, T(1), T(2)), T(5))\n    assert is_subtree(big, T(4, T(1), T(2))) and not is_subtree(big, T(4, T(1))) and is_subtree(big, T(5)) and is_subtree(big, None)\n    assert not is_subtree(T(12), T(2)), \"values must be separated\"\n    assert not is_subtree(T(3, T(4, T(1), T(2, T(0))), T(5)), T(4, T(1), T(2)))\ndef test_big():\n    \"20,000 nodes\"\n    rng = random.Random(1)\n    nodes = [T(rng.randint(0, 3)) for _ in range(20_000)]\n    for i in range(1, 20_000):\n        parent = nodes[(i - 1) // 2]\n        if i % 2:\n            parent.left = nodes[i]\n        else:\n            parent.right = nodes[i]\n    assert is_subtree(nodes[0], nodes[777]) and not is_subtree(nodes[0], T(9))")

### Exercise: Merging two trees

Write `merge(a, b)`: a **new** tree whose shape is the union of both, where each node holds the sum of the values at that position in `a` and `b` (a missing node counts as 0). Don't change `a` or `b`, and don't share nodes with them.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def merge(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef levels(n):\n    from collections import deque\n    out, q = [], deque([n])\n    while q:\n        x = q.popleft()\n        out.append(None if x is None else x.val)\n        if x is not None:\n            q.extend([x.left, x.right])\n    while out and out[-1] is None:\n        out.pop()\n    return out\ndef ids(n):\n    return set() if n is None else {id(n)} | ids(n.left) | ids(n.right)\ndef test_merge():\n    \"merging\"\n    a = T(1, T(3, T(5)), T(2))\n    b = T(2, T(1, None, T(4)), T(3, None, T(7)))\n    m = merge(a, b)\n    assert levels(m) == [3, 4, 5, 5, 4, None, 7]\n    assert levels(a) == [1, 3, 2, 5] and not (ids(m) & (ids(a) | ids(b))), \"make new nodes\"\n    assert merge(None, None) is None and levels(merge(None, T(1, T(2)))) == [1, 2]")

### Bug hunt: Stopping too early

This checks the remaining sum when it reaches `None`, so a path that stops at a node with only one child counts as a root-to-leaf path: for target 1 it says `True`, using just the root of `1 → 2`. Check at **leaves** instead, so it prints `False True`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def has_path_sum(node, target):
    if node is None:
        return target == 0
    return has_path_sum(node.left, target - node.val) or has_path_sum(node.right, target - node.val)

t = TreeNode(1, TreeNode(2))
print(has_path_sum(t, 1), has_path_sum(t, 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"leaves only\"\n    expect_output(\"False True\")")

### Bug hunt: Saving a path that keeps changing

This saves the **same** list object for every path it finds, and backtracking empties that list afterwards, so every saved path ends up empty: this prints `[[], []]`. Save a copy, so it prints `[[1, 2], [1, 3]]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def all_paths(root):
    found, path = [], []
    def walk(node):
        if node is None:
            return
        path.append(node.val)
        if node.left is None and node.right is None:
            found.append(path)
        walk(node.left)
        walk(node.right)
        path.pop()
    walk(root)
    return found

print(all_paths(TreeNode(1, TreeNode(2), TreeNode(3))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"copy the path\"\n    expect_output(\"[[1, 2], [1, 3]]\")")

### Bug hunt: Updated too soon

This updates the highest value **before** comparing, so each node is compared with a maximum that already includes itself, and `>` is never true: this prints `0`. Compare first (with `>=`), then update, so it prints `4`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def good_nodes(node, highest=float("-inf")):
    if node is None:
        return 0
    highest = max(highest, node.val)
    good = 1 if node.val > highest else 0
    return good + good_nodes(node.left, highest) + good_nodes(node.right, highest)

T = TreeNode
print(good_nodes(T(3, T(1, T(3)), T(4, T(1), T(5)))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"compare, then update\"\n    expect_output(\"4\")")

### Puzzle: A root-to-leaf path sum

Arrange the lines into a function checking for a root-to-leaf path with a given sum. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# For the tree         1
#                     / \
#                    2   3
#                   / \
#                  4   5

p = ___   # the root-to-leaf sums, left to right
q = ___   # the numbers spelled by the root-to-leaf paths, left to right
r = ___   # the sum of all subtree sums (each node's subtree sum, added up)
s = ___   # the number of good nodes (no bigger value above them)
t = ___   # "down" or "up": which way the information flows in good_nodes

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [7, 8, 4] and q == [124, 125, 13] and r == 38, \"p, q and r\"\n    assert s == 5 and t == \"down\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: counting paths with a sum

Write `count_paths(root, k)`: the number of **downward** paths (starting at any node and going down through children, ending at any node at or below it; a single node is a path) whose values add up to `k`. Values may be negative. Do it in O(n) with a dictionary of prefix sums on the current path, adding each node's prefix on the way down and removing it on the way back up. Use an **explicit stack**, not recursion: the checks use a chain 20,000 nodes tall. (Hint: push each node twice, once to enter it and once to leave it, as Day 3's marked post-order did.)

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def count_paths(root, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef slow(root, k):\n    def from_here(n, need):\n        if n is None:\n            return 0\n        need -= n.val\n        return (need == 0) + from_here(n.left, need) + from_here(n.right, need)\n    def every(n):\n        if n is None:\n            return 0\n        return from_here(n, k) + every(n.left) + every(n.right)\n    return every(root)\ndef random_tree(rng, n):\n    root = T(rng.randint(-3, 3))\n    nodes = [root]\n    while len(nodes) < n:\n        p = rng.choice(nodes)\n        c = T(rng.randint(-3, 3))\n        if p.left is None:\n            p.left = c\n        elif p.right is None:\n            p.right = c\n        else:\n            continue\n        nodes.append(c)\n    return root\ndef test_small():\n    \"small\"\n    t = T(10, T(5, T(3, T(3), T(-2)), T(2, None, T(1))), T(-3, None, T(11)))\n    assert count_paths(t, 8) == 3 and count_paths(None, 0) == 0 and count_paths(T(0, T(0)), 0) == 3\n    rng = random.Random(5)\n    for _ in range(100):\n        t = random_tree(rng, rng.randint(1, 20))\n        k = rng.randint(-4, 4)\n        assert count_paths(t, k) == slow(t, k)\ndef test_tall():\n    \"a tall chain\"\n    chain = None\n    for i in range(20_000):\n        chain = T(1 if i % 2 else -1, chain)\n    assert count_paths(chain, 0) == 100_000_000")

## Warm-up

Each call is told the sum of the path **above** it (`so_far`), adds its own value, and passes that down. At a leaf, the total is a complete root-to-leaf sum. Information flowed **down** the tree, through a parameter.

## Information flowing down

When the answer at a node depends on what's **above** it (the path so far, the depth, the largest value seen on the way down), pass it down as a parameter. Each call receives the state from its parent and hands an updated copy to each child.

Because numbers and strings can't be changed, each child gets its own value and nothing needs undoing. With a **list** (the path itself), either pass a new list to each child (`path + [node.val]`) or append, recurse, and `pop()` afterwards: that undoing step is called **backtracking**, and forgetting it is the classic bug.

## Information flowing up

When the answer at a node depends on what's **below** it, compute it from the children's answers (post-order, Day 3). The helper returns whatever the parent needs, which may be more than the final answer.

In `count_unival`, both children are always visited (`left, right = ...`) before combining. Writing `same(node.left) and same(node.right)` would skip the right side whenever the left fails, and miss its subtrees.

## Both ways

Some problems need information in both directions. "Sum of root-to-leaf numbers" treats each path as the digits of a number (the path 1 → 2 → 3 is 123): the number so far flows **down** (`so_far * 10 + node.val`), and the totals of the leaves flow **up**.

The pattern to remember: **parameters carry information down; return values carry it up.**

## Whole trees

Some problems compare or combine entire trees.

**Is one tree a subtree of another?** The simple way checks `same(node, sub)` (Day 2) at every node: O(n × m). A neat O(n + m) way uses Day 3's serialisation: save both trees with markers for missing children, and check whether one string occurs inside the other. The markers matter: without them, different shapes could give the same text. (Wrap each value in separators, as below, so that 2 isn't found inside 12.)

(Python's `in` on strings is fast in practice; Day 1 of last week's Rabin and Karp search guarantees it.)

**Merging two trees**: walk both at once; where both have a node, make one with the sum; where only one does, use that one's subtree as it is.

## Paths and prefix sums

How many **downward** paths (starting at any node, going down through children, ending anywhere below) add up to `k`? Checking every start node and every path below it is O(n²) for a tall tree. Week 41's prefix sums do it in O(n): walking down from the root, keep a dictionary of the prefix sums **on the current path**; a path ending at this node with sum `k` exists for each earlier prefix equal to `current − k`. When you leave a node, take its prefix sum back out of the dictionary (backtracking), because it's no longer on the path.

Without the `seen[current] -= 1`, prefix sums from a finished branch would still count when the walk moves into a different branch, pairing nodes that aren't on the same path.

## Practice

## Quiz

## Daily challenge

## Before you go

For each problem today, write one line: does information flow down, up, or both, and what does each call pass or return? Tomorrow is the Forge, where a file system's folders, a family tree and an expression all become trees.